# Conference-Style Results Summary

This notebook summarizes `testscript.py` outputs into compact paper-ready artifacts:
1. Main summary table (overall by agent)
2. Scenario robustness heatmap (composite rank)
3. Pareto tradeoff scatter (allocation vs handovers)
4. Critical-case time series (hard scenarios only)

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


BASE_DIR = Path('.').resolve()
AGENTS = ['BASELINE', 'PPO', 'DQN', 'ODT', 'ORACLE',  'ORACLE_DROP', 'ORACLE_LATENCY', 'ODT_FINETUNED']
SCENARIOS = [
    'load_cycle_1',
    'load_cycle_2',
    'load_cycle_5',
    'medium_aircraft',
    'snr_congested',
]

LATENCY_CLIP_MS = 1000  # for optional filtered latency displays

print('Base dir:', BASE_DIR)

Base dir: /Users/hindmukhtar/Documents/GitHub/DLRL2025/Single Constellation 


In [2]:
def load_runs(base_dir: Path, agents, scenarios):
    rows = []
    missing = []

    # Ensure we point to the folder containing the CSVs
    base_dir = Path(base_dir)
    if not (base_dir / "BASELINE_observations_no_scenario.csv").exists():
        candidate = base_dir / "Single Constellation "
        if candidate.exists():
            base_dir = candidate

    for agent in agents:
        for scenario in scenarios:
            # Try expected file first
            candidates = [
                base_dir / f"{agent}_observations_{scenario}.csv",
            ]


            fp = next((p for p in candidates if p.exists()), None)
            if fp is None:
                missing.append(f"{agent}_observations_{scenario}.csv")
                continue

            df = pd.read_csv(fp)

            # Normalize headers from CSVs written with spaces after commas
            df.columns = (
                df.columns
                .str.strip()
                .str.replace(r"\s+", "_", regex=True)
            )
            df["agent"] = agent
            df["scenario"] = scenario

            rows.append(df)

    data = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    return data, missing, base_dir

data, missing, used_dir = load_runs(BASE_DIR, AGENTS, SCENARIOS)
print("Using dir:", used_dir)
print("Loaded rows:", len(data))
print("Loaded agent/scenario pairs:",
      data[["agent", "scenario"]].drop_duplicates().shape[0] if len(data) else 0)
if missing:
    print("Missing files (if any):", ", ".join(missing[:12]), "..." if len(missing) > 12 else "")
data.head()


Using dir: /Users/hindmukhtar/Documents/GitHub/DLRL2025/Single Constellation 
Loaded rows: 43280
Loaded agent/scenario pairs: 40


,step,lat,lon,alt,snr,load,handovers,allocated_bw,allocation_ratio,demand_MB,...,queing_delay_s,propagation_latency_s,transmission_rate_mbps,latency_req_s,beam_capacity,service_drop_s,dwell_remaining_s,ttt_remaining_s,agent,scenario
0,0,29.956900,-95.336899,327.660004,18.503132,0.659110,0.0,0.562500,1.0,0.562500,...,0.0,0.052687,4.5,0.2,60.000000,0.0,4.0,0.0,BASELINE,load_cycle_1
1,1,29.954182,-95.333305,365.760010,8.830511,0.658336,0.0,3.070534,1.0,3.070534,...,0.0,0.053876,4.5,0.2,327.523621,0.0,0.0,0.0,BASELINE,load_cycle_1
2,2,29.951462,-95.329712,403.859985,8.818870,0.667468,0.0,2.986333,1.0,2.986333,...,0.0,0.053095,4.5,0.2,318.542236,0.0,0.0,0.0,BASELINE,load_cycle_1
3,3,29.948744,-95.326118,441.959991,17.458019,0.678836,1.0,2.906630,1.0,2.906630,...,0.0,0.052824,4.5,0.2,310.040527,0.0,0.0,0.0,BASELINE,load_cycle_1
4,4,29.945999,-95.322273,472.440002,9.066678,0.664004,1.0,2.934174,1.0,2.934174,...,0.0,0.052417,4.5,0.2,312.978577,0.0,0.0,0.0,BASELINE,load_cycle_1


In [3]:
data['latency_s'] = data['queing_delay_s'] + data['propagation_latency_s']

In [4]:
def episode_metrics(df):
    out = []
    for (agent, scenario), g in df.groupby(['agent', 'scenario']):
        # Active-demand slice for fair latency/allocation statistics.
        if 'demand_MB' in g.columns:
            g_active = g[g['demand_MB'] > 0].copy()
        else:
            g_active = g.copy()

        if g_active.empty:
            alloc_ratio = 1.0
            lat_violation_rate = np.nan
            avg_latency_ms = np.nan
            mean_excess_ms = np.nan
            p95_excess_ms = np.nan
            severity_score = np.nan
        else:
            alloc_ratio = float(g_active['allocation_ratio'].mean())
            # Treat sentinel queue delay (1000 s) as outage, not a valid latency sample.
            if 'queing_delay_s' in g_active.columns:
                g_active.loc[g_active['queing_delay_s'] >= 1000, 'latency_s'] = np.nan

            lat_req = g_active['latency_req_s'] if 'latency_req_s' in g_active.columns else pd.Series([np.inf] * len(g_active), index=g_active.index)
            valid_lat = g_active['latency_s'].notna()
            if valid_lat.any():
                lat_s = g_active.loc[valid_lat, 'latency_s']
                req_s = lat_req.loc[valid_lat]
                excess_s = (lat_s - req_s).clip(lower=0)
                lat_violation_rate = float((excess_s > 0).mean())
                avg_latency_ms = float(lat_s.mean() * 1000.0)
                mean_excess_ms = float(excess_s.mean() * 1000.0)
                p95_excess_ms = float(excess_s.quantile(0.95) * 1000.0)
                severity_score = float((excess_s / req_s).mean())
            else:
                lat_violation_rate = np.nan
                avg_latency_ms = np.nan
                mean_excess_ms = np.nan
                p95_excess_ms = np.nan
                severity_score = np.nan

        service_drop_s = float(g['service_drop_s'].sum()) if 'service_drop_s' in g.columns else 0.0
        total_handovers = float(g['handovers'].max()) if 'handovers' in g.columns else np.nan

        # Composite score: higher is better.
        J = (
            1.0 * alloc_ratio
            #  - 0.01 * lat_violation_rate
            # -service_drop_s
        )

        out.append({
            'agent': agent,
            'scenario': scenario,
            'allocation_ratio': alloc_ratio,
            'latency_violation_rate': lat_violation_rate,
            'service_drop_s': service_drop_s,
            'total_handovers': total_handovers,
            'avg_latency_ms': avg_latency_ms,
            'mean_excess_ms': mean_excess_ms,
            'p95_excess_ms': p95_excess_ms,
            'severity_score': severity_score,
            'composite_score': J,
        })
    return pd.DataFrame(out)

m = episode_metrics(data)
m.sort_values(['scenario','composite_score'], ascending=[True, False]).head(20)

,agent,scenario,allocation_ratio,latency_violation_rate,service_drop_s,total_handovers,avg_latency_ms,mean_excess_ms,p95_excess_ms,severity_score,composite_score
25,ORACLE_DROP,load_cycle_1,1.000000,0.007394,0.000000,258.0,54.320751,0.398315,0.0,inf,1.000000
30,ORACLE_LATENCY,load_cycle_1,1.000000,0.007394,0.000000,258.0,54.320751,0.398315,0.0,inf,1.000000
35,PPO,load_cycle_1,1.000000,0.007394,0.000000,230.0,54.783544,0.416044,0.0,inf,1.000000
5,DQN,load_cycle_1,0.998405,0.009242,0.000000,250.0,112.833105,58.033789,0.0,inf,0.998405
10,ODT,load_cycle_1,0.998405,0.009242,0.000000,251.0,112.842661,58.035345,0.0,inf,0.998405
15,ODT_FINETUNED,load_cycle_1,0.998405,0.009242,0.000000,251.0,112.796291,58.035345,0.0,inf,0.998405
20,ORACLE,load_cycle_1,0.993784,0.009285,24.688061,256.0,113.064933,58.306928,0.0,inf,0.993784
0,BASELINE,load_cycle_1,0.984542,0.009372,74.010345,257.0,113.596730,58.853385,0.0,inf,0.984542
1,BASELINE,load_cycle_2,0.998071,0.008333,9.569111,261.0,55.282847,0.761357,0.0,inf,0.998071
6,DQN,load_cycle_2,0.998071,0.008333,9.569111,249.0,55.288553,0.757041,0.0,inf,0.998071


## 1) Main Summary Table (Overall by Agent)

In [5]:
overall = (
    m.groupby('agent', as_index=False)
     .agg({
         'allocation_ratio': 'mean',
         'latency_violation_rate': 'mean',
         'mean_excess_ms': 'mean',
         'p95_excess_ms': 'mean',
         'severity_score': 'mean',
         'service_drop_s': 'mean',
         'total_handovers': 'mean',
         'avg_latency_ms': 'mean',
         'composite_score': 'mean',
     })
)
overall = overall.sort_values('composite_score', ascending=False)
overall.style.format({
    'allocation_ratio': '{:.4f}',
    'latency_violation_rate': '{:.2%}',
    'mean_excess_ms': '{:.2f}',
    'p95_excess_ms': '{:.2f}',
    'severity_score': '{:.4f}',
    'service_drop_s': '{:.2f}',
    'total_handovers': '{:.1f}',
    'avg_latency_ms': '{:.2f}',
    'composite_score': '{:.4f}',
})

,agent,allocation_ratio,latency_violation_rate,mean_excess_ms,p95_excess_ms,severity_score,service_drop_s,total_handovers,avg_latency_ms,composite_score
5,ORACLE_DROP,0.9941,1.54%,12.82,0.00,nan,30.27,254.2,66.53,0.9941
6,ORACLE_LATENCY,0.9941,1.54%,12.82,0.00,nan,30.27,254.2,66.53,0.9941
3,ODT_FINETUNED,0.9918,1.69%,76.90,0.00,nan,36.27,250.2,131.19,0.9918
2,ODT,0.9916,1.69%,76.92,0.00,nan,37.29,249.8,131.21,0.9916
1,DQN,0.9905,1.70%,77.24,0.00,nan,42.95,247.8,131.53,0.9905
4,ORACLE,0.9901,1.81%,78.94,9.59,nan,44.16,255.2,133.26,0.9901
7,PPO,0.9874,1.80%,68.16,9.64,nan,60.00,245.8,122.41,0.9874
0,BASELINE,0.9834,1.86%,88.42,9.64,nan,78.75,256.8,142.78,0.9834


In [6]:
import numpy as np
import pandas as pd

df = data.copy()

# ---- Robust derived fields ----
if "allocation_ratio" not in df.columns:
    demand = pd.to_numeric(df.get("demand_MB", 0.0), errors="coerce").fillna(0.0).clip(lower=0)
    alloc = pd.to_numeric(df.get("allocated_bw", 0.0), errors="coerce").fillna(0.0).clip(lower=0)
    df["allocation_ratio"] = np.where(demand > 1e-9, alloc / demand, 1.0)
    df["allocation_ratio"] = np.clip(df["allocation_ratio"], 0.0, 1.0)

if "latency_s" not in df.columns:
    q = pd.to_numeric(df.get("queing_delay_s", 0.0), errors="coerce").fillna(0.0)
    p = pd.to_numeric(df.get("propagation_latency_s", 0.0), errors="coerce").fillna(0.0)
    df["latency_s"] = q + p

if "queing_delay_s" in df.columns:
    q = pd.to_numeric(df["queing_delay_s"], errors="coerce")
    df.loc[q >= 1000, "latency_s"] = np.nan

if "throughput_mbps" in df.columns:
    thr_col = "throughput_mbps"
elif "allocated_bw" in df.columns:
    thr_col = "allocated_bw"
elif "transmission_rate_mbps" in df.columns:
    thr_col = "transmission_rate_mbps"
else:
    raise ValueError("No throughput-like column found.")

# Active-demand steps only
active = df[pd.to_numeric(df.get("demand_MB", 0.0), errors="coerce").fillna(0.0) > 0].copy()

m = (
    active.groupby(["scenario", "agent"], as_index=False)
    .agg(
        avg_allocation_ratio=("allocation_ratio", "mean"),
        total_service_drop_s=("service_drop_s", "sum"),
        avg_throughput_mbps=(thr_col, "mean"),
        avg_latency_ms=("latency_s", lambda s: np.nanmean(s) * 1000.0),
    )
)

# Rank by allocation ratio (higher is better) within each scenario
m["rank_by_alloc"] = (
    m.groupby("scenario")["avg_allocation_ratio"]
     .rank(ascending=False, method="min")
     .astype(int)
)

# Long format -> pivot with agents as columns
long = m.melt(
    id_vars=["scenario", "agent"],
    value_vars=["rank_by_alloc", "avg_allocation_ratio", "total_service_drop_s", "avg_throughput_mbps", "avg_latency_ms"],
    var_name="metric",
    value_name="value"
)

# Optional friendly names
metric_name = {
    "rank_by_alloc": "Rank (by allocation ratio)",
    "avg_allocation_ratio": "Avg allocation/demand",
    "total_service_drop_s": "Total service drop (s)",
    "avg_throughput_mbps": "Avg throughput (Mbps)",
    "avg_latency_ms": "Avg latency (ms)",
}
long["metric"] = long["metric"].map(metric_name)

table = (
    long.pivot_table(index=["scenario", "metric"], columns="agent", values="value", aggfunc="first")
        .sort_index()
)

display(
    table.style.format({
        col: "{:.0f}" if "Rank" in str(table.index.get_level_values(1)[0]) else "{:.4f}"
        for col in table.columns
    }).format("{:.0f}", subset=pd.IndexSlice[pd.IndexSlice[:, "Rank (by allocation ratio)"], :])
      .format("{:.4f}", subset=pd.IndexSlice[pd.IndexSlice[:, "Avg allocation/demand"], :])
      .format("{:.1f}", subset=pd.IndexSlice[pd.IndexSlice[:, "Total service drop (s)"], :])
      .format("{:.2f}", subset=pd.IndexSlice[pd.IndexSlice[:, "Avg throughput (Mbps)"], :])
      .format("{:.2f}", subset=pd.IndexSlice[pd.IndexSlice[:, "Avg latency (ms)"], :])
)


## 4) Critical-Case Time Series (Hard Scenarios Only)

In [7]:
SCENARIO = "snr_congested"
plot_df = data[data["scenario"] == SCENARIO].copy()

if len(plot_df) == 0:
    print(f"No data found for scenario: {SCENARIO}")
else:
    plot_df = plot_df.sort_values(["agent", "step"])

    plot_df["throughput_alloc_mbps"] = pd.to_numeric(
        plot_df.get("transmission_rate_mbps", np.nan), errors="coerce"
    )

    ref_agent = sorted(plot_df["agent"].unique())[0]
    req_df = plot_df[plot_df["agent"] == ref_agent].copy()
    req_df["throughput_req_mbps"] = pd.to_numeric(
        req_df.get("throughput_req", np.nan), errors="coerce"
    )

    plot_df["alloc_smooth"] = (
        plot_df.groupby("agent")["throughput_alloc_mbps"]
               .transform(lambda s: s.rolling(15, min_periods=1).mean())
    )
    req_df["req_smooth"] = req_df["throughput_req_mbps"].rolling(15, min_periods=1).mean()

    fig = go.Figure()

    # 1) Add allocated first
    for agent in sorted(plot_df["agent"].unique()):
        g = plot_df[plot_df["agent"] == agent]
        if g.empty:
            continue
        fig.add_trace(
            go.Scatter(
                x=g["step"],
                y=g["alloc_smooth"],
                mode="lines",
                name=f"{agent}",
            )
        )

    # 2) Add requested last so it draws on top
    fig.add_trace(
        go.Scatter(
            x=req_df["step"],
            y=req_df["req_smooth"],
            mode="lines",
            name="Requested Throughput",
            line=dict(color="black", dash="dash", width=3),
        )
    )

    fig.update_layout(
        height=520,
        title=f"Allocated vs Requested Throughput (High SNR, High Congestion)",
        xaxis_title="Step",
        yaxis_title="Throughput (Mbps)",
        legend=dict(
            orientation="h",
            yanchor="top",
            y=-0.2,
            xanchor="center",
            x=0.5
        ),
        legend_title="Trace",
    )
    fig.show()


In [8]:
SCENARIO = "medium_aircraft"
plot_df = data[data["scenario"] == SCENARIO].copy()

if len(plot_df) == 0:
    print(f"No data found for scenario: {SCENARIO}")
else:
    plot_df = plot_df.sort_values(["agent", "step"])

    plot_df["throughput_alloc_mbps"] = pd.to_numeric(
        plot_df.get("transmission_rate_mbps", np.nan), errors="coerce"
    )

    ref_agent = sorted(plot_df["agent"].unique())[0]
    req_df = plot_df[plot_df["agent"] == ref_agent].copy()
    req_df["throughput_req_mbps"] = pd.to_numeric(
        req_df.get("throughput_req", np.nan), errors="coerce"
    )

    plot_df["alloc_smooth"] = (
        plot_df.groupby("agent")["throughput_alloc_mbps"]
               .transform(lambda s: s.rolling(15, min_periods=1).mean())
    )
    req_df["req_smooth"] = req_df["throughput_req_mbps"].rolling(15, min_periods=1).mean()

    fig = go.Figure()

    for agent in sorted(plot_df["agent"].unique()):
        g = plot_df[plot_df["agent"] == agent]
        if g.empty:
            continue
        fig.add_trace(
            go.Scatter(
                x=g["step"],
                y=g["alloc_smooth"],
                mode="lines",
                name=agent,
            )
        )

    fig.add_trace(
        go.Scatter(
            x=req_df["step"],
            y=req_df["req_smooth"],
            mode="lines",
            name="Requested Throughput",
            line=dict(color="black", dash="dash", width=3),
        )
    )

    fig.update_layout(
        height=520,
        title=f"Allocated vs Requested Throughput (Large Aircraft)",
        xaxis_title="Step",
        yaxis_title="Throughput (Mbps)",
        legend_title="Trace",
        legend=dict(
            orientation="h",
            yanchor="top",
            y=-0.2,
            xanchor="center",
            x=0.5
        ),
        margin=dict(b=100),
    )
    fig.show()
